# Phase 0 — Auto Ball Tracking Validation

Run order: top to bottom. Use **Runtime → Change runtime type → T4 GPU** before starting.

Outputs `tracks.json` + `preview_tv.mp4` + `preview_compare.mp4`. Watch the preview videos to decide go/no-go on the full pipeline.

## 1. Install dependencies

In [2]:
# If running on the local Mac venv (~/.venvs/match-tracker-phase0), deps are already installed — this cell is a no-op.
# To re-install via corporate artifactory (Mac/Apple Silicon, Python 3.12), uncomment:
# %pip install -q --index-url https://artifactory.foc.zone/artifactory/api/pypi/rdf-pypi-virtual/simple --extra-index-url https://artifactory.foc.zone/artifactory/api/pypi/pypi-remote/simple "torch>=2.2,<2.6" "torchvision<0.21" opencv-python-headless "numpy<2" tqdm pillow pyyaml requests
# %pip install -q --no-deps --index-url https://artifactory.foc.zone/artifactory/api/pypi/rdf-pypi-virtual/simple --extra-index-url https://artifactory.foc.zone/artifactory/api/pypi/pypi-remote/simple ultralytics ultralytics-thop py-cpuinfo

import torch, platform

if torch.cuda.is_available():
    DEVICE = 'cuda'
    name = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    name = f'Apple Silicon MPS ({platform.machine()})'
else:
    DEVICE = 'cpu'
    name = 'CPU'
print(f'Device: {DEVICE} — {name}')


Device: mps — Apple Silicon MPS (arm64)


## 2. Provide input video

Either upload a clip from your Insta360 X5 (stitched equirectangular MP4), or set `INPUT_URL` to a sample. Keep clips ≤ 60 sec for fast iteration.

In [4]:
import os, cv2

# Local Mac run: point at the SouthArm X5 sample sitting next to this notebook.
INPUT_PATH = os.path.join(os.path.dirname(os.path.abspath('')), 'tracking', 'samples', 'southarmfc_x5_60sec.mp4')
if not os.path.isfile(INPUT_PATH):
    # Fallback: try relative to current working directory
    candidate = 'samples/southarmfc_x5_60sec.mp4'
    if os.path.isfile(candidate):
        INPUT_PATH = os.path.abspath(candidate)
assert os.path.isfile(INPUT_PATH), f'Sample not found at {INPUT_PATH}. Run tracking/samples/fetch_sample.sh first.'

OUT_DIR = os.path.join(os.path.dirname(INPUT_PATH), '..', 'outputs')
OUT_DIR = os.path.abspath(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(INPUT_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# True 2:1 equirectangular vs YouTube-repacked 16:9 (or any other aspect).
# YouTube serves 360° videos in 16:9 containers with spherical metadata, so the
# Phase 0 pipeline runs in "flat" mode: YOLO detection + naive crop/pan render.
IS_EQUIRECTANGULAR = abs(W / H - 2.0) < 0.05

print(f'Input:  {INPUT_PATH}')
print(f'Format: {W}x{H} @ {FPS:.1f}fps, {N} frames, {N/FPS:.1f}s')
print(f'Output: {OUT_DIR}')
print(f'Mode:   {"equirectangular (true 2:1)" if IS_EQUIRECTANGULAR else "flat (non-2:1; perspective render will use crop/pan)"}')


Input:  /Users/irezaeian/match-tracker/tracking/samples/southarmfc_x5_60sec.mp4
Format: 2560x1440 @ 30.0fps, 1880 frames, 62.7s
Output: /Users/irezaeian/match-tracker/tracking/outputs
Mode:   flat (non-2:1; perspective render will use crop/pan)


## 3. Load YOLOv8 detector

Using `yolov8n.pt` (smallest, fastest). COCO-pretrained → classes 0=person, 32=sports ball. Good enough for validation; we fine-tune later if needed.

In [5]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.to(DEVICE)
print(f'Model loaded on {DEVICE}.')

100%|██████████| 6.25M/6.25M [00:02<00:00, 3.25MB/s]


Model loaded on mps.


## 4. Run detection per frame

**Strategy for equirectangular**: action stays near the equator (mid 33% vertically) on a level-tripod 360° rig. We crop that band before YOLO to (a) reduce distortion and (b) speed up inference 3x.

Coordinate output: `lon ∈ [-180, 180]`, `lat ∈ [-90, 90]`. Lon=0 is the direction the camera was facing at start.

In [6]:
from tqdm import tqdm
import numpy as np

# Process at 10 Hz (every ~3 frames at 30fps) — matches output sample rate
SAMPLE_HZ = 10
stride = max(1, round(FPS / SAMPLE_HZ))

# Equator band crop (middle 33% vertically) — only when truly equirectangular.
# For flat (16:9) input the field already fills the frame, so use the full frame.
if IS_EQUIRECTANGULAR:
    BAND_TOP = int(H * 0.33)
    BAND_BOT = int(H * 0.67)
else:
    BAND_TOP = 0
    BAND_BOT = H
BAND_H = BAND_BOT - BAND_TOP

def px_to_lonlat(cx, cy):
    # For 2:1 equirectangular: full sphere mapping.
    # For flat input: scale x → [-180, 180] across the frame, y → [-90, 90].
    # This keeps the same downstream code (smoothing in degrees) without crashing.
    lon = (cx / W) * 360.0 - 180.0
    lat = 90.0 - (cy / H) * 180.0
    return lon, lat

cap = cv2.VideoCapture(INPUT_PATH)
raw_detections = []  # list of dicts per processed frame

frame_idx = 0
pbar = tqdm(total=N, desc='Detecting')
while True:
    ret, frame = cap.read()
    if not ret: break
    if frame_idx % stride == 0:
        band = frame[BAND_TOP:BAND_BOT, :, :]
        # Resize for speed (keep aspect)
        det_w = 1280
        det_h = int(BAND_H * det_w / W)
        small = cv2.resize(band, (det_w, det_h))
        results = model.predict(small, conf=0.25, classes=[0, 32], verbose=False)
        players, ball = [], None
        if len(results) > 0:
            r = results[0]
            for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(),
                                       r.boxes.cls.cpu().numpy(),
                                       r.boxes.conf.cpu().numpy()):
                x1, y1, x2, y2 = box
                cx = (x1 + x2) / 2 * (W / det_w)
                cy_in_band = (y1 + y2) / 2 * (BAND_H / det_h)
                cy = BAND_TOP + cy_in_band
                lon, lat = px_to_lonlat(cx, cy)
                if int(cls) == 0:  # person
                    players.append({'lon': lon, 'lat': lat, 'conf': float(conf)})
                elif int(cls) == 32 and (ball is None or conf > ball['conf']):
                    ball = {'lon': lon, 'lat': lat, 'conf': float(conf)}
        raw_detections.append({
            't': frame_idx / FPS,
            'players': players,
            'ball': ball,
        })
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()
print(f'Processed {len(raw_detections)} sampled frames @ {SAMPLE_HZ}Hz')
print(f'Frames with ball detection: {sum(1 for d in raw_detections if d["ball"])} / {len(raw_detections)}')
print(f'Avg players per frame: {np.mean([len(d["players"]) for d in raw_detections]):.1f}')


Detecting: 100%|██████████| 1880/1880 [00:22<00:00, 82.54it/s] 

Processed 627 sampled frames @ 10Hz
Frames with ball detection: 0 / 627
Avg players per frame: 0.3


## 5. Compute action centroid + smooth

Fuse ball + player cluster into a single target point. Apply EMA + dead zone + lead. Output the camera state stream.

In [7]:
EMA_ALPHA = 0.15           # lower = smoother, higher = more responsive
BALL_WEIGHT = 0.55         # weight of ball vs player centroid (when ball detected)
LEAD_FRAMES = 6            # how many samples ahead to lead the camera
DEAD_ZONE_DEG = 8.0        # don't move if target within this of current
FOV_TIGHT = 35.0
FOV_WIDE = 65.0

def angular_diff(a, b):
    d = (a - b + 180) % 360 - 180
    return d

def circular_mean(values):
    if not values: return None
    rads = np.deg2rad(values)
    s = np.mean(np.sin(rads)); c = np.mean(np.cos(rads))
    return float(np.rad2deg(np.arctan2(s, c)))

raw_targets = []
for d in raw_detections:
    players = d['players']
    ball = d['ball']
    if not players and not ball:
        raw_targets.append(None)
        continue
    player_lon = circular_mean([p['lon'] for p in players]) if players else None
    player_lat = float(np.mean([p['lat'] for p in players])) if players else None
    if ball and player_lon is not None:
        # weighted mix
        diff = angular_diff(ball['lon'], player_lon)
        tgt_lon = (player_lon + BALL_WEIGHT * diff) % 360
        if tgt_lon > 180: tgt_lon -= 360
        tgt_lat = (1 - BALL_WEIGHT) * player_lat + BALL_WEIGHT * ball['lat']
    elif ball:
        tgt_lon, tgt_lat = ball['lon'], ball['lat']
    else:
        tgt_lon, tgt_lat = player_lon, player_lat
    # FOV: tight if players spread is small, wide if spread is large
    if len(players) > 1:
        lons = np.array([p['lon'] for p in players])
        spread = np.std(lons)
        fov = float(np.clip(FOV_TIGHT + spread * 1.5, FOV_TIGHT, FOV_WIDE))
    else:
        fov = FOV_TIGHT + 10
    raw_targets.append({'t': d['t'], 'lon': tgt_lon, 'lat': tgt_lat, 'fov': fov, 'hasBall': ball is not None})

# Forward-fill gaps (lost detections) with last known
last = None
filled = []
for r in raw_targets:
    if r is None and last is not None:
        filled.append(dict(last))
    else:
        filled.append(r)
        if r is not None: last = r

# EMA smooth (circular for lon)
smoothed = []
sm_lon = sm_lat = sm_fov = None
for r in filled:
    if r is None: continue
    if sm_lon is None:
        sm_lon, sm_lat, sm_fov = r['lon'], r['lat'], r['fov']
    else:
        # Apply dead zone
        d_lon = angular_diff(r['lon'], sm_lon)
        d_lat = r['lat'] - sm_lat
        if abs(d_lon) > DEAD_ZONE_DEG or abs(d_lat) > DEAD_ZONE_DEG * 0.5:
            sm_lon = (sm_lon + EMA_ALPHA * d_lon) % 360
            if sm_lon > 180: sm_lon -= 360
            sm_lat = sm_lat + EMA_ALPHA * d_lat
        sm_fov = sm_fov + EMA_ALPHA * (r['fov'] - sm_fov)
    smoothed.append({'t': r['t'], 'lon': sm_lon, 'lat': sm_lat, 'fov': sm_fov})

# Apply lead (shift future N samples back)
led = []
for i, s in enumerate(smoothed):
    j = min(len(smoothed) - 1, i + LEAD_FRAMES)
    led.append({
        't': s['t'],
        'lon': smoothed[j]['lon'],
        'lat': smoothed[j]['lat'],
        'fov': smoothed[j]['fov'],
    })

print(f'Smoothed track: {len(led)} samples')

Smoothed track: 554 samples


## 6. Write tracks.json (the contract for the client)

In [8]:
import json

tracks = {
    'fps': SAMPLE_HZ,
    'duration': N / FPS,
    'trackingVersion': 1,
    'samples': [[round(s['t'], 3), round(s['lon'], 2), round(s['lat'], 2), round(s['fov'], 1)] for s in led]
}
TRACKS_PATH = os.path.join(OUT_DIR, 'tracks.json')
with open(TRACKS_PATH, 'w') as f:
    json.dump(tracks, f, separators=(',', ':'))
size_kb = os.path.getsize(TRACKS_PATH) / 1024
print(f'Wrote {TRACKS_PATH}: {size_kb:.1f} KB ({len(led)} samples)')
print('Per-minute size:', round(size_kb / (N/FPS/60), 1), 'KB/min')

Wrote /Users/irezaeian/match-tracker/tracking/outputs/tracks.json: 13.5 KB (554 samples)
Per-minute size: 12.9 KB/min


## 7. Render TV-mode preview (THE VERDICT)

Re-projects the equirectangular source through a virtual camera following the smoothed track. Output: `preview_tv.mp4`.

**Watch this video. If it feels watchable → ship it. If not → tune params (EMA_ALPHA, DEAD_ZONE_DEG, LEAD_FRAMES) and re-run from step 5.**

In [9]:
OUT_W, OUT_H = 1280, 720  # 16:9 TV crop

def render_perspective(equi, lon_deg, lat_deg, fov_deg, out_w, out_h):
    """Sample equirectangular `equi` through a perspective camera. Returns BGR image."""
    fh, fw = equi.shape[:2]
    # Build ray grid
    fov = np.deg2rad(fov_deg)
    f = (out_w / 2) / np.tan(fov / 2)
    cx, cy = out_w / 2, out_h / 2
    xs = np.arange(out_w) - cx
    ys = np.arange(out_h) - cy
    xx, yy = np.meshgrid(xs, ys)
    zs = np.full_like(xx, f, dtype=np.float32)
    dirs = np.stack([xx, yy, zs], axis=-1).astype(np.float32)
    norms = np.linalg.norm(dirs, axis=-1, keepdims=True)
    dirs /= norms
    # Rotate by lat (around X), then lon (around Y)
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)
    cl, sl = np.cos(lat), np.sin(lat)
    co, so = np.cos(lon), np.sin(lon)
    Rx = np.array([[1,0,0],[0,cl,-sl],[0,sl,cl]], dtype=np.float32)
    Ry = np.array([[co,0,so],[0,1,0],[-so,0,co]], dtype=np.float32)
    R = Ry @ Rx
    rotated = dirs @ R.T
    x, y, z = rotated[..., 0], rotated[..., 1], rotated[..., 2]
    # Convert to equirectangular UVs
    theta = np.arctan2(x, z)
    phi = np.arcsin(np.clip(y, -1, 1))
    u = (theta / (2 * np.pi) + 0.5) * fw
    v = (phi / np.pi + 0.5) * fh
    return cv2.remap(equi, u.astype(np.float32), v.astype(np.float32),
                     interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)

def lerp_track(t, samples):
    # Binary search
    import bisect
    times = [s[0] for s in samples]
    i = bisect.bisect_left(times, t)
    if i == 0: return samples[0][1], samples[0][2], samples[0][3]
    if i >= len(samples): return samples[-1][1], samples[-1][2], samples[-1][3]
    t0, lon0, lat0, fov0 = samples[i-1]
    t1, lon1, lat1, fov1 = samples[i]
    a = (t - t0) / max(1e-6, t1 - t0)
    # Circular lerp for lon
    dlon = ((lon1 - lon0 + 180) % 360) - 180
    lon = lon0 + a * dlon
    lat = lat0 + a * (lat1 - lat0)
    fov = fov0 + a * (fov1 - fov0)
    return lon, lat, fov

samples = tracks['samples']
TV_PATH = os.path.join(OUT_DIR, 'preview_tv.mp4')
CMP_PATH = os.path.join(OUT_DIR, 'preview_compare.mp4')
cap = cv2.VideoCapture(INPUT_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_tv = cv2.VideoWriter(TV_PATH, fourcc, FPS, (OUT_W, OUT_H))
out_cmp = cv2.VideoWriter(CMP_PATH, fourcc, FPS, (OUT_W, OUT_H * 2))

frame_idx = 0
pbar = tqdm(total=N, desc='Rendering')
while True:
    ret, frame = cap.read()
    if not ret: break
    t = frame_idx / FPS
    lon, lat, fov = lerp_track(t, samples)
    tv = render_perspective(frame, lon, lat, fov, OUT_W, OUT_H)
    out_tv.write(tv)
    # Side-by-side: top = wide equirectangular thumbnail, bottom = TV crop
    wide_thumb = cv2.resize(frame, (OUT_W, OUT_H))
    # Draw indicator of where TV camera is pointing on the wide view
    cx_ind = int((lon + 180) / 360 * OUT_W)
    cy_ind = int((90 - lat) / 180 * OUT_H)
    cv2.rectangle(wide_thumb, (cx_ind-30, cy_ind-20), (cx_ind+30, cy_ind+20), (0, 255, 0), 2)
    stacked = np.vstack([wide_thumb, tv])
    out_cmp.write(stacked)
    frame_idx += 1
    pbar.update(1)
pbar.close()
cap.release()
out_tv.release()
out_cmp.release()
print(f'Done.\n  TV preview:      {TV_PATH}\n  Compare preview: {CMP_PATH}')

Rendering: 100%|██████████| 1880/1880 [01:38<00:00, 19.00it/s]

Done.
  TV preview:      /Users/irezaeian/match-tracker/tracking/outputs/preview_tv.mp4
  Compare preview: /Users/irezaeian/match-tracker/tracking/outputs/preview_compare.mp4


## 7b. 4K vs 5.7K comparison (decide recording resolution)

Re-runs **detection only** (no smoothing, no render) on the same video downscaled to a 4K-equivalent equirectangular (3840×1920). Compares:

- **Ball detection recall** — % of frames where YOLO found the ball at each resolution
- **Position agreement** — average angular error between 4K and 5.7K ball positions on frames where both detected the ball

If 4K recall is within ~5% of 5.7K and angular agreement is <3°, **record at 4K** to save 4× upload time + storage. Otherwise stay at 5.7K.

In [10]:
# Compare ball detection at native 5.7K vs downscaled-to-4K equirectangular.
# Reuses the same equator-band + YOLO pipeline from cell 4, just at two input sizes.

def detect_pass(target_w):
    """Run detection on the video resized to (target_w, target_w//2). Returns per-sample ball lon/lat or None."""
    target_h = target_w // 2
    band_top = int(target_h * 0.33)
    band_bot = int(target_h * 0.67)
    band_h = band_bot - band_top
    cap = cv2.VideoCapture(INPUT_PATH)
    out = []
    fi = 0
    pbar = tqdm(total=N, desc=f'Detect @ {target_w}x{target_h}', leave=False)
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fi % stride == 0:
            if frame.shape[1] != target_w:
                frame_r = cv2.resize(frame, (target_w, target_h))
            else:
                frame_r = frame
            band = frame_r[band_top:band_bot, :, :]
            det_w = 1280
            det_h = int(band_h * det_w / target_w)
            small = cv2.resize(band, (det_w, det_h))
            results = model.predict(small, conf=0.25, classes=[32], verbose=False)
            ball = None
            if len(results) > 0:
                r = results[0]
                for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(),
                                           r.boxes.cls.cpu().numpy(),
                                           r.boxes.conf.cpu().numpy()):
                    if int(cls) != 32: continue
                    x1, y1, x2, y2 = box
                    cx = (x1 + x2) / 2 * (target_w / det_w)
                    cy_in_band = (y1 + y2) / 2 * (band_h / det_h)
                    cy = band_top + cy_in_band
                    lon = (cx / target_w) * 360.0 - 180.0
                    lat = 90.0 - (cy / target_h) * 180.0
                    if ball is None or conf > ball['conf']:
                        ball = {'lon': lon, 'lat': lat, 'conf': float(conf)}
            out.append(ball)
        fi += 1
        pbar.update(1)
    pbar.close()
    cap.release()
    return out

# Native resolution (whatever the input was — typically 5.7K = 5760x2880)
native_w = W if W % 2 == 0 else W - 1
det_native = detect_pass(native_w)

# 4K-equivalent equirectangular = 3840x1920
det_4k = detect_pass(3840)

n = min(len(det_native), len(det_4k))
hits_native = sum(1 for b in det_native[:n] if b is not None)
hits_4k = sum(1 for b in det_4k[:n] if b is not None)

# Angular error on frames where BOTH detected the ball
errs = []
for a, b in zip(det_native[:n], det_4k[:n]):
    if a is None or b is None: continue
    dlon = ((a['lon'] - b['lon'] + 180) % 360) - 180
    dlat = a['lat'] - b['lat']
    errs.append(float(np.hypot(dlon, dlat)))

print(f'\n=== 4K vs {native_w}x{native_w//2} comparison ===')
print(f'Frames sampled:           {n}')
print(f'Ball recall @ native:     {hits_native}/{n} = {hits_native/n*100:.1f}%')
print(f'Ball recall @ 4K:         {hits_4k}/{n} = {hits_4k/n*100:.1f}%')
print(f'Recall delta (native-4K): {(hits_native - hits_4k)/n*100:+.1f} pp')
if errs:
    print(f'Position agreement:       mean {np.mean(errs):.2f}°, median {np.median(errs):.2f}°, p95 {np.percentile(errs, 95):.2f}° (n={len(errs)})')
else:
    print('Position agreement:       no co-detected frames')

# Verdict
recall_delta = (hits_native - hits_4k) / n * 100
med_err = float(np.median(errs)) if errs else 999
print('\nRECOMMENDATION:')
if recall_delta <= 5 and med_err < 3:
    print('  ✅ Record at 4K — saves ~4× storage/upload, minimal detection loss.')
elif recall_delta <= 10 and med_err < 5:
    print('  ⚠️  Borderline — record 5.7K if upload time is acceptable, else 4K.')
else:
    print('  ❌ Stay at 5.7K — 4K loses too much ball detail at your camera distance.')


=== 4K vs 2560x1280 comparison ===
Frames sampled:           627
Ball recall @ native:     0/627 = 0.0%
Ball recall @ 4K:         0/627 = 0.0%
Recall delta (native-4K): +0.0 pp
Position agreement:       no co-detected frames

RECOMMENDATION:
  ❌ Stay at 5.7K — 4K loses too much ball detail at your camera distance.


## 8. Download outputs

In [11]:
# Outputs are already on disk at OUT_DIR. Open them in Finder / QuickTime:
import subprocess
print(f'Outputs in: {OUT_DIR}')
for f in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, f)
    print(f'  {f:30s}  {os.path.getsize(full)/1024:.1f} KB')

# Uncomment to auto-open in Finder:
# subprocess.run(['open', OUT_DIR])

Outputs in: /Users/irezaeian/match-tracker/tracking/outputs
  preview_compare.mp4             45548.9 KB
  preview_tv.mp4                  15902.2 KB
  tracks.json                     13.5 KB


## What to do with the results

1. Watch `preview_tv.mp4` end-to-end — does it feel like TV?
2. Watch `preview_compare.mp4` — does the green box on the wide view track sensibly?
3. Note any moments where:
   - Camera lags behind action → reduce `EMA_ALPHA` or increase `LEAD_FRAMES`
   - Camera jitters → increase `DEAD_ZONE_DEG`
   - Camera misses a goal or break → likely a detection gap; consider fine-tuning YOLO on your footage
4. Inspect `tracks.json` shape — confirm the schema is what the client needs:
   ```json
   {
     "fps": 10,
     "duration": 60.0,
     "trackingVersion": 1,
     "samples": [[t, lon, lat, fov], ...]
   }
   ```